# 01. Od procedury do funkcji z Gemini

[![Otwórz w Colabie](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caqdastm/ai_qda-workshop-1u/blob/main/00_github_colab/01_colab_gemini_vibe_coding.ipynb)

**Odznaka otwiera wzorzec** z repozytorium prowadzących. Do trwałej pracy otwórz notebook z własnego prywatnego repo utworzonego z szablonu.

Tutaj zobaczysz, jak opis procedury w języku badawczym zostaje
połączony z przygotowanym kontekstem technicznym. Gemini ma napisać
tylko jedną małą funkcję. Twoim zadaniem jest ocena procedury i jej
wyniku, nie ręczne programowanie.


## Rytm pracy

**Cel pracy z materiałem -> karta procedury -> prompt z dodatkiem
technicznym -> kod od modelu -> lista kontroli -> inspekcja
analityczna -> decyzja badacza.**

Kontrola techniczna odpowiada na pytanie „czy funkcja zrobiła to, co
zapisaliśmy w kontrakcie?”. Nie odpowiada na pytanie „czy rezultat
jest trafny interpretacyjnie?”.


In [2]:
# @title Sprawdź prywatny workspace — uruchom bez edycji { display-mode: "form" }
from pathlib import Path
import base64
import os
import re
import subprocess
import sys

PUBLIC_REPOSITORY_URL = "https://github.com/caqdastm/ai_qda-workshop-1u.git"
PUBLIC_REPOSITORY_SLUG = "caqdastm/ai_qda-workshop-1u"
REPOSITORY_REF = "main"

def _colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

PARTICIPANT_REPOSITORY = str(
    _colab_secret("AI_QDA_REPOSITORY") or ""
).strip().rstrip("/")
if PARTICIPANT_REPOSITORY.endswith(".git"):
    PARTICIPANT_REPOSITORY = PARTICIPANT_REPOSITORY[:-4]
if PARTICIPANT_REPOSITORY.startswith("https://github.com/"):
    PARTICIPANT_REPOSITORY = PARTICIPANT_REPOSITORY.removeprefix(
        "https://github.com/"
    )
if PARTICIPANT_REPOSITORY and not re.fullmatch(
    r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", PARTICIPANT_REPOSITORY
):
    raise ValueError(
        "Sekret AI_QDA_REPOSITORY podaj jako login/nazwa-repozytorium."
    )
if PARTICIPANT_REPOSITORY.lower() == PUBLIC_REPOSITORY_SLUG.lower():
    raise ValueError(
        "AI_QDA_REPOSITORY musi wskazywać Twoje prywatne repo, "
        "nie repo prowadzących."
    )

GITHUB_TOKEN = _colab_secret("GITHUB_TOKEN")
if PARTICIPANT_REPOSITORY:
    if not GITHUB_TOKEN:
        raise RuntimeError(
            "Włącz dla tego notebooka dostęp do sekretu GITHUB_TOKEN."
        )
    REPOSITORY_URL = f"https://github.com/{PARTICIPANT_REPOSITORY}.git"
    WORKSPACE_MODE = "participant_repository"
else:
    REPOSITORY_URL = PUBLIC_REPOSITORY_URL
    WORKSPACE_MODE = "public_demo"

REPO_ROOT = Path("/content/ai_qda_workshop_workspace")
if not (REPO_ROOT / "04_vibe_coding" / "workshop_support.py").is_file():
    clone_environment = os.environ.copy()
    if GITHUB_TOKEN:
        encoded = base64.b64encode(
            f"x-access-token:{GITHUB_TOKEN}".encode("utf-8")
        ).decode("ascii")
        clone_environment["GIT_CONFIG_COUNT"] = "1"
        clone_environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
        clone_environment["GIT_CONFIG_VALUE_0"] = (
            f"AUTHORIZATION: basic {encoded}"
        )
    clone_result = subprocess.run(
        [
            "git", "clone", "--depth", "1", "--branch", REPOSITORY_REF,
            REPOSITORY_URL, str(REPO_ROOT),
        ],
        env=clone_environment,
        capture_output=True,
        text=True,
    )
    if clone_result.returncode != 0:
        raise RuntimeError(
            "Nie udało się otworzyć repozytorium roboczego. Sprawdź "
            "sekrety AI_QDA_REPOSITORY i GITHUB_TOKEN oraz dostęp tokenu "
            f"do repo. Git: {clone_result.stderr.strip()}"
        )

support_dir = REPO_ROOT / "04_vibe_coding"
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))
from workshop_support import (
    publish_outputs_to_github,
    read_secret,
    save_dataframe,
    save_json,
)

INTRO_OUTPUTS_DIR = REPO_ROOT / "00_github_colab" / "outputs"
INTRO_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print("Tryb workspace:", WORKSPACE_MODE)
print("Repo uczestnika:", PARTICIPANT_REPOSITORY or "brak — publiczny tryb demonstracyjny")
print("Wyniki wprowadzenia:", INTRO_OUTPUTS_DIR)


Tryb workspace: participant_repository
Repo uczestnika: DamianS-V/warsztat_ai_coding
Wyniki wprowadzenia: /content/ai_qda_workshop_workspace/00_github_colab/outputs


## 1. Obejrzyj materiał przed projektowaniem procedury

Poniższa mini-transkrypcja została utworzona wyłącznie do ćwiczenia.
Zwróć uwagę na dokładny tekst, kolejność oraz etykiety mówców. Jedna
etykieta jest celowo niejednoznaczna.

Pytanie do pary: **co mogłoby zostać utracone albo zbyt szybko
rozstrzygnięte podczas automatycznego przygotowania tej rozmowy?**


In [3]:
# @title Infrastruktura: przygotuj mini-transkrypcję { display-mode: "form" }
import pandas as pd
from IPython.display import display

mini_transcript = pd.DataFrame(
    [
        {"interview_id": "WYWIAD_01", "source_row": 1, "speaker_raw": "Badacz", "text": "Co zmieniło się w Pani pracy w ostatnim roku?"},
        {"interview_id": "WYWIAD_01", "source_row": 2, "speaker_raw": "Respondent", "text": "Najtrudniejsze było ciągłe przełączanie się między zadaniami."},
        {"interview_id": "WYWIAD_01", "source_row": 3, "speaker_raw": "Badacz", "text": "Co to dla Pani oznaczało na co dzień?"},
        {"interview_id": "WYWIAD_01", "source_row": 4, "speaker_raw": "Respondent", "text": "Miałam poczucie, że niczego nie kończę naprawdę dobrze."},
        {"interview_id": "WYWIAD_02", "source_row": 1, "speaker_raw": "Moderator", "text": "Jak zespół reagował na nowe zasady?"},
        {"interview_id": "WYWIAD_02", "source_row": 2, "speaker_raw": "Uczestniczka", "text": "Na początku każdy próbował radzić sobie osobno."},
        {"interview_id": "WYWIAD_02", "source_row": 3, "speaker_raw": "Moderator", "text": "A kiedy pojawiła się współpraca?"},
        {"interview_id": "WYWIAD_02", "source_row": 4, "speaker_raw": "Rozmówczyni", "text": "Dopiero gdy wspólnie nazwaliśmy problem."},
    ]
)
source_before = mini_transcript.copy(deep=True)


In [4]:
# @title Pokaż materiał wejściowy { display-mode: "form" }
display(mini_transcript)
print("Etykiety mówców:", ", ".join(mini_transcript["speaker_raw"].unique()))


,interview_id,source_row,speaker_raw,text
0,WYWIAD_01,1,Badacz,Co zmieniło się w Pani pracy w ostatnim roku?
1,WYWIAD_01,2,Respondent,Najtrudniejsze było ciągłe przełączanie się mi...
2,WYWIAD_01,3,Badacz,Co to dla Pani oznaczało na co dzień?
3,WYWIAD_01,4,Respondent,"Miałam poczucie, że niczego nie kończę naprawd..."
4,WYWIAD_02,1,Moderator,Jak zespół reagował na nowe zasady?
5,WYWIAD_02,2,Uczestniczka,Na początku każdy próbował radzić sobie osobno.
6,WYWIAD_02,3,Moderator,A kiedy pojawiła się współpraca?
7,WYWIAD_02,4,Rozmówczyni,Dopiero gdy wspólnie nazwaliśmy problem.


Etykiety mówców: Badacz, Respondent, Moderator, Uczestniczka, Rozmówczyni


Zanim przejdziesz dalej, dokończ ustnie dwa zdania:

- „Procedura ma pomóc mi ...”
- „Procedura nie powinna samodzielnie rozstrzygać ...”

W dalszym ćwiczeniu przygotujemy jednostki materiału, zachowując
ślad do źródła. Niejednoznaczną etykietę skierujemy do przeglądu
zamiast automatycznie przypisywać jej rolę.


In [5]:
# @title 2. Karta procedury — edytuj język badawczy, nie kod { display-mode: "form" }
cel_procedury = "Przygotować jednostki materiału do dalszego kodowania bez utraty związku ze źródłem." # @param {type:"string"}
oczekiwany_rezultat = "Każdy wiersz ma stabilne ID i rolę mówcy; dokładny tekst oraz kolejność są zachowane." # @param {type:"string"}
warunki_kontroli = "Nie ubywa wierszy, tekst i kolejność się nie zmieniają, a identyfikatory są unikalne." # @param {type:"string"}
granica_automatyzacji = "Niejednoznaczna etykieta otrzymuje needs_review; jej interpretację rozstrzyga badacz." # @param {type:"string"}

PROCEDURE_CARD = {
    "goal": cel_procedury,
    "observable_result": oczekiwany_rezultat,
    "automatic_checks": warunki_kontroli,
    "researcher_decision": granica_automatyzacji,
}


In [6]:
# @title 3. Infrastruktura: zbuduj prompt dla Gemini { display-mode: "form" }
RESEARCH_PART = f"""
CZĘŚĆ BADAWCZA — intencja i granice procedury
Cel: {PROCEDURE_CARD['goal']}
Widoczny rezultat: {PROCEDURE_CARD['observable_result']}
Kontrole: {PROCEDURE_CARD['automatic_checks']}
Decyzja badacza: {PROCEDURE_CARD['researcher_decision']}
""".strip()

TECHNICAL_APPENDIX = """
DODATEK TECHNICZNY — przygotowane dopasowanie do notebooka
Napisz jedną funkcję prepare_evidence_units_ai(df).
Wejście jest tabelą pandas z kolumnami:
interview_id, source_row, speaker_raw, text.
Zwróć nową tabelę z zachowanymi kolumnami i wierszami oraz:
- evidence_id: interview_id + _U + trzycyfrowa kolejność w wywiadzie,
- speaker_role: moderator, participant albo needs_review.
Moderator: M, Moderator, Badacz.
Uczestnik: R, Respondent, Uczestniczka.
Każda inna etykieta ma otrzymać needs_review.
Nie zmieniaj df w miejscu. Jeśli brakuje wymaganej kolumny,
zgłoś ValueError z nazwami brakujących kolumn.
Zwróć wyłącznie kod funkcji, bez danych przykładowych.
""".strip()

PROMPT_FOR_GEMINI = RESEARCH_PART + "\n\n" + TECHNICAL_APPENDIX
print(PROMPT_FOR_GEMINI)


CZĘŚĆ BADAWCZA — intencja i granice procedury
Cel: Przygotować jednostki materiału do dalszego kodowania bez utraty związku ze źródłem.
Widoczny rezultat: Każdy wiersz ma stabilne ID i rolę mówcy; dokładny tekst oraz kolejność są zachowane.
Kontrole: Nie ubywa wierszy, tekst i kolejność się nie zmieniają, a identyfikatory są unikalne.
Decyzja badacza: Niejednoznaczna etykieta otrzymuje needs_review; jej interpretację rozstrzyga badacz.

DODATEK TECHNICZNY — przygotowane dopasowanie do notebooka
Napisz jedną funkcję prepare_evidence_units_ai(df).
Wejście jest tabelą pandas z kolumnami:
interview_id, source_row, speaker_raw, text.
Zwróć nową tabelę z zachowanymi kolumnami i wierszami oraz:
- evidence_id: interview_id + _U + trzycyfrowa kolejność w wywiadzie,
- speaker_role: moderator, participant albo needs_review.
Moderator: M, Moderator, Badacz.
Uczestnik: R, Respondent, Uczestniczka.
Każda inna etykieta ma otrzymać needs_review.
Nie zmieniaj df w miejscu. Jeśli brakuje wymaganej kolum

## 4. Poproś model o implementację

1. Otwórz panel Gemini w Colabie.
2. Skopiuj cały prompt wyświetlony powyżej.
3. Zwróć uwagę, co pochodzi z Twojej karty procedury, a co dodała
   infrastruktura warsztatowa.
4. Wklej otrzymaną funkcję do następnej komórki.

Nie proś o cały notebook ani pipeline. Na tym etapie implementujemy
jeden ograniczony krok.


In [7]:
import pandas as pd

def prepare_evidence_units_ai(df):
    required_columns = {"interview_id", "source_row", "speaker_raw", "text"}
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Brak wymaganych kolumn: {', '.join(missing_columns)}")

    result_df = df.copy()

    # Assign speaker_role
    speaker_role_mapping = {
        'm': 'moderator',
        'moderator': 'moderator',
        'badacz': 'moderator',
        'r': 'participant',
        'respondent': 'participant',
        'uczestniczka': 'participant'
    }
    result_df['speaker_role'] = result_df['speaker_raw'].astype(str).str.lower().map(speaker_role_mapping).fillna('needs_review')

    # Generate evidence_id
    result_df['sequence_in_interview'] = result_df.groupby('interview_id').cumcount() + 1
    result_df['evidence_id'] = result_df['interview_id'] + '_U' + result_df['sequence_in_interview'].astype(str).str.zfill(3)

    return result_df


In [ ]:
# @title Rozwiązanie awaryjne — uruchom, jeśli funkcja nie jest dostępna { display-mode: "form" }
def prepared_fallback(df):
    required = {"interview_id", "source_row", "speaker_raw", "text"}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Brak wymaganych kolumn: {missing}")

    result = df.copy()
    role_map = {
        "m": "moderator",
        "moderator": "moderator",
        "badacz": "moderator",
        "r": "participant",
        "respondent": "participant",
        "uczestniczka": "participant",
    }
    normalized = result["speaker_raw"].astype(str).str.strip().str.lower()
    result["speaker_role"] = normalized.map(role_map).fillna("needs_review")
    order = result.groupby("interview_id", sort=False).cumcount() + 1
    result["evidence_id"] = (
        result["interview_id"].astype(str)
        + "_U"
        + order.astype(str).str.zfill(3)
    )
    return result

if not callable(globals().get("prepare_evidence_units_ai")):
    prepare_evidence_units_ai = prepared_fallback
    print("Używasz przygotowanego rozwiązania awaryjnego.")
else:
    print("Używasz funkcji wygenerowanej w rozmowie z Gemini.")


In [8]:
# @title 5. Uruchom procedurę i przeczytaj listę kontroli { display-mode: "form" }
try:
    prepared_units = prepare_evidence_units_ai(mini_transcript)
except Exception as error:
    print("Procedura nie uruchomiła się. Przekaż modelowi ten komunikat:")
    print(type(error).__name__ + ":", error)
    prepared_units = None

required_output = {
    "interview_id", "source_row", "speaker_raw", "text",
    "speaker_role", "evidence_id",
}
is_table = isinstance(prepared_units, pd.DataFrame)
has_columns = is_table and required_output.issubset(prepared_units.columns)
ambiguous_rows = (
    prepared_units.loc[
        (prepared_units["interview_id"] == "WYWIAD_02")
        & (prepared_units["source_row"] == 4)
    ]
    if has_columns else pd.DataFrame()
)

checks = [
    {"kontrola": "Zachowano liczbę jednostek", "zaliczona": is_table and len(prepared_units) == len(mini_transcript), "znaczenie": "Żaden fragment nie zniknął ani nie został dodany."},
    {"kontrola": "Zachowano wymagane informacje", "zaliczona": has_columns, "znaczenie": "Można wrócić do źródła i zobaczyć wynik procedury."},
    {"kontrola": "Tekst pozostał dokładnie taki sam", "zaliczona": has_columns and prepared_units["text"].tolist() == mini_transcript["text"].tolist(), "znaczenie": "Procedura nie przepisała wypowiedzi."},
    {"kontrola": "Kolejność źródłowa została zachowana", "zaliczona": has_columns and prepared_units["source_row"].tolist() == mini_transcript["source_row"].tolist(), "znaczenie": "Sekwencja rozmowy nie została przestawiona."},
    {"kontrola": "Identyfikatory są kompletne i unikalne", "zaliczona": has_columns and prepared_units["evidence_id"].notna().all() and prepared_units["evidence_id"].is_unique, "znaczenie": "Każdą jednostkę można jednoznacznie wskazać."},
    {"kontrola": "Niejednoznaczna etykieta czeka na przegląd", "zaliczona": len(ambiguous_rows) == 1 and ambiguous_rows.iloc[0]["speaker_role"] == "needs_review", "znaczenie": "Program nie zastąpił decyzji badacza automatycznym przypisaniem."},
    {"kontrola": "Dane wejściowe nie zostały zmienione", "zaliczona": mini_transcript.equals(source_before), "znaczenie": "Procedura utworzyła nowy rezultat."},
]
checks_table = pd.DataFrame(checks)
display(checks_table)
if prepared_units is not None:
    display(prepared_units)

failed = checks_table.loc[~checks_table["zaliczona"], "kontrola"].tolist()
if failed:
    print("Wymagają poprawy:", "; ".join(failed))
else:
    print("Wszystkie uzgodnione kontrole techniczne są spełnione.")
    print("Teraz potrzebna jest inspekcja i decyzja badacza.")


,kontrola,zaliczona,znaczenie
0,Zachowano liczbę jednostek,True,Żaden fragment nie zniknął ani nie został dodany.
1,Zachowano wymagane informacje,True,Można wrócić do źródła i zobaczyć wynik proced...
2,Tekst pozostał dokładnie taki sam,True,Procedura nie przepisała wypowiedzi.
3,Kolejność źródłowa została zachowana,True,Sekwencja rozmowy nie została przestawiona.
4,Identyfikatory są kompletne i unikalne,True,Każdą jednostkę można jednoznacznie wskazać.
5,Niejednoznaczna etykieta czeka na przegląd,True,Program nie zastąpił decyzji badacza automatyc...
6,Dane wejściowe nie zostały zmienione,True,Procedura utworzyła nowy rezultat.


,interview_id,source_row,speaker_raw,text,speaker_role,sequence_in_interview,evidence_id
0,WYWIAD_01,1,Badacz,Co zmieniło się w Pani pracy w ostatnim roku?,moderator,1,WYWIAD_01_U001
1,WYWIAD_01,2,Respondent,Najtrudniejsze było ciągłe przełączanie się mi...,participant,2,WYWIAD_01_U002
2,WYWIAD_01,3,Badacz,Co to dla Pani oznaczało na co dzień?,moderator,3,WYWIAD_01_U003
3,WYWIAD_01,4,Respondent,"Miałam poczucie, że niczego nie kończę naprawd...",participant,4,WYWIAD_01_U004
4,WYWIAD_02,1,Moderator,Jak zespół reagował na nowe zasady?,moderator,1,WYWIAD_02_U001
5,WYWIAD_02,2,Uczestniczka,Na początku każdy próbował radzić sobie osobno.,participant,2,WYWIAD_02_U002
6,WYWIAD_02,3,Moderator,A kiedy pojawiła się współpraca?,moderator,3,WYWIAD_02_U003
7,WYWIAD_02,4,Rozmówczyni,Dopiero gdy wspólnie nazwaliśmy problem.,needs_review,4,WYWIAD_02_U004


Wszystkie uzgodnione kontrole techniczne są spełnione.
Teraz potrzebna jest inspekcja i decyzja badacza.


## 6. Inspekcja analityczna

Zielona lista nie kończy pracy. Obejrzyj wynik i odpowiedz:

- Czy jednostka wiersza jest użyteczna dla planowanego kodowania?
- Czy stabilny identyfikator pozwala wrócić do dokładnego miejsca?
- Która etykieta trafiła do needs_review i jakiego kontekstu
  potrzebujesz do jej rozstrzygnięcia?
- Co zmieniłoby się, gdyby każdą nieznaną etykietę automatycznie
  uznać za uczestnika?
- Jaki jeden warunek doprecyzujesz przed przejściem na korpus?

Jeśli zmieniasz procedurę, zmień jedno wymaganie w karcie, poproś
Gemini o aktualizację tej samej funkcji i ponownie uruchom kontrole.


In [9]:
# @title 7. Zapisz decyzję badacza { display-mode: "form" }
sposob_obslugi_niejednoznacznosci = "pozostaw needs_review" # @param ["pozostaw needs_review", "przypisz automatycznie po udokumentowanej regule"]
potrzebny_kontekst = "Dokumentacja etykiet mówców i podgląd sąsiednich wypowiedzi." # @param {type:"string"}
ocena_jednostki = "Wiersz zachowuje sekwencję, ale jego przydatność dla kodowania trzeba ocenić na dłuższym fragmencie." # @param {type:"string"}
zmiana_przed_korpusem = "Dodać udokumentowaną mapę etykiet występujących w korpusie i kolejkę przypadków do przeglądu." # @param {type:"string"}

RESEARCHER_DECISIONS = pd.DataFrame(
    [
        {"pytanie": "Jak obsłużyć niejednoznaczność?", "decyzja": sposob_obslugi_niejednoznacznosci},
        {"pytanie": "Jaki kontekst jest potrzebny?", "decyzja": potrzebny_kontekst},
        {"pytanie": "Czy jednostka jest użyteczna?", "decyzja": ocena_jednostki},
        {"pytanie": "Co zmienić przed korpusem?", "decyzja": zmiana_przed_korpusem},
    ]
)
display(RESEARCHER_DECISIONS)


,pytanie,decyzja
0,Jak obsłużyć niejednoznaczność?,pozostaw needs_review
1,Jaki kontekst jest potrzebny?,Dokumentacja etykiet mówców i podgląd sąsiedni...
2,Czy jednostka jest użyteczna?,"Wiersz zachowuje sekwencję, ale jego przydatno..."
3,Co zmienić przed korpusem?,Dodać udokumentowaną mapę etykiet występującyc...


In [10]:
# @title 8. Infrastruktura: zapisz rezultat i decyzje { display-mode: "form" }
units_path = INTRO_OUTPUTS_DIR / "01_prepared_evidence_units.csv"
checks_path = INTRO_OUTPUTS_DIR / "01_quality_report.csv"
decisions_path = INTRO_OUTPUTS_DIR / "01_researcher_decisions.csv"
card_path = INTRO_OUTPUTS_DIR / "01_procedure_card.json"

save_dataframe(units_path, prepared_units)
save_dataframe(checks_path, checks_table)
save_dataframe(decisions_path, RESEARCHER_DECISIONS)
save_json(card_path, PROCEDURE_CARD)
print("Zapisano:", units_path)
print("Zapisano:", checks_path)
print("Zapisano:", decisions_path)
print("Zapisano:", card_path)


Zapisano: /content/ai_qda_workshop_workspace/00_github_colab/outputs/01_prepared_evidence_units.csv
Zapisano: /content/ai_qda_workshop_workspace/00_github_colab/outputs/01_quality_report.csv
Zapisano: /content/ai_qda_workshop_workspace/00_github_colab/outputs/01_researcher_decisions.csv
Zapisano: /content/ai_qda_workshop_workspace/00_github_colab/outputs/01_procedure_card.json


In [11]:
# @title Zapisz sprawdzone wyniki w swoim repo GitHub { display-mode: "form" }
PUBLISH_RESULTS_TO_GITHUB = True # @param {type:"boolean"}

if PUBLISH_RESULTS_TO_GITHUB:
    if WORKSPACE_MODE != "participant_repository":
        raise RuntimeError(
            "Dodaj w Colab Secrets AI_QDA_REPOSITORY i GITHUB_TOKEN, "
            "włącz ich dostęp i uruchom notebook ponownie od początku."
        )
    publication = publish_outputs_to_github(
        REPO_ROOT,
        [INTRO_OUTPUTS_DIR],
        message='AI QDA: zapisz pierwszą procedurę',
        participant_repository=PARTICIPANT_REPOSITORY,
        token=read_secret("GITHUB_TOKEN"),
        branch=REPOSITORY_REF,
        include_api_logs=False,
    )
    print("Zapis GitHub:", publication["status"])
    print("Commit:", publication.get("commit", "bez nowej zmiany"))
    print("Pliki:", publication["paths"])
else:
    print(
        "Wyniki są tylko w runtime. Po kontroli ustaw "
        "PUBLISH_RESULTS_TO_GITHUB=True i uruchom komórkę ponownie."
    )


Zapis GitHub: pushed
Commit: d8704b35b25c4e6508d92a97afae4519aea0cc74
Pliki: ['00_github_colab/outputs/01_prepared_evidence_units.csv', '00_github_colab/outputs/01_procedure_card.json', '00_github_colab/outputs/01_quality_report.csv', '00_github_colab/outputs/01_researcher_decisions.csv']


## 9. Zapis notebooka i produktów

Najpierw przeczytaj zapisane tabele i ustaw
`PUBLISH_RESULTS_TO_GITHUB=True`. Sprawdź status `pushed` lub
`up_to_date` oraz pliki w `00_github_colab/outputs/` prywatnego repo.

Notebook zapisz osobno przez **Plik → Zapisz kopię w GitHubie**. Dzięki
temu prywatne repo zachowuje zarówno procedurę, jak i jej sprawdzone
produkty. Klucze i tokeny pozostają wyłącznie w Colab Secrets.
